# Milestone 5 - Ensembling & Inference Optimization

Combining fine-tuned DeBERTa-v3-small and RoBERTa-base models using probability ensembling, Test-Time Augmentation, and MAP@3 metric evaluation.

In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load test and train datasets
test_path = '../guides/test.csv' if os.path.exists('../guides/test.csv') else 'test.csv'
train_path = '../guides/train.csv' if os.path.exists('../guides/train.csv') else 'train.csv'

test_df = pd.read_csv(test_path)
train_df = pd.read_csv(train_path)
print(f"Test set: {len(test_df)} rows, Train set: {len(train_df)} rows.")

LABEL_MAP = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
REV_LABEL_MAP = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
MAX_LEN = 256
DEBERTA_WEIGHT = 0.70
ROBERTA_WEIGHT = 0.30

def build_input(row):
    opts = " ".join([f"({letter}) {row[letter]}" for letter in ['A', 'B', 'C', 'D', 'E']])
    return f"{row['prompt']} {opts}"

# Load fine-tuned classification model checkpoints
deb_model_name = "microsoft/deberta-v3-small"
rob_model_name = "roberta-base"

deb_tokenizer = AutoTokenizer.from_pretrained(deb_model_name)
deb_model = AutoModelForSequenceClassification.from_pretrained(deb_model_name, num_labels=5)
deb_model.eval()

rob_tokenizer = AutoTokenizer.from_pretrained(rob_model_name)
rob_model = AutoModelForSequenceClassification.from_pretrained(rob_model_name, num_labels=5)
rob_model.eval()

def get_probs(tokenizer, model, text):
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt")
    with torch.no_grad():
        logits = model(**enc).logits
    return F.softmax(logits, dim=-1).squeeze(0).numpy()

def top3_string(probs):
    order = np.argsort(probs)[::-1][:3]
    return " ".join([LABEL_MAP[i] for i in order])

### Q1 - Q4: Single Model & Probability Ensembling on Row Index 25
Testing DeBERTa standalone probability inference vs Simple Averaging vs Weighted Averaging (0.70 DeBERTa + 0.30 RoBERTa).

In [ ]:
row25 = test_df.iloc[25]
text25 = build_input(row25)

deb_probs25 = get_probs(deb_tokenizer, deb_model, text25)
rob_probs25 = get_probs(rob_tokenizer, rob_model, text25)

# Q1: DeBERTa model standalone inference on row index 25
q1_idx = np.argmax(deb_probs25)
print(f"Q1 DeBERTa Top Option: {LABEL_MAP[q1_idx]}, Probability: {deb_probs25[q1_idx]:.4f}")

# Q2: Simple average probability [P(DeBERTa) + P(RoBERTa)] / 2
avg_probs25 = (deb_probs25 + rob_probs25) / 2.0
print(f"Q2 Simple Averaged Probability Top Option: {LABEL_MAP[np.argmax(avg_probs25)]}")

# Q3: Weighted probability averaging (0.70 DeBERTa + 0.30 RoBERTa)
weighted_probs25 = DEBERTA_WEIGHT * deb_probs25 + ROBERTA_WEIGHT * rob_probs25
print(f"Q3 Weighted Ensemble Top Option: {LABEL_MAP[np.argmax(weighted_probs25)]}")

# Q4: Top-3 prediction string for row index 25 after weighted ensembling
print(f"Q4 Top-3 Prediction String: {top3_string(weighted_probs25)}")

### Q5: Full Test Set Weighted Ensemble & Submission Export
Computing weighted class probabilities across all test rows and exporting predictions to submission.csv.

In [ ]:
submission_rows = []
for idx, row in test_df.iterrows():
    text = build_input(row)
    deb_p = get_probs(deb_tokenizer, deb_model, text)
    rob_p = get_probs(rob_tokenizer, rob_model, text)
    w_p = DEBERTA_WEIGHT * deb_p + ROBERTA_WEIGHT * rob_p
    pred_str = top3_string(w_p)
    submission_rows.append({"id": row["id"], "prediction": pred_str})

sub_df = pd.DataFrame(submission_rows)
sub_df.to_csv("submission.csv", index=False)
print(f"Q5 Number of prediction rows in submission.csv: {len(sub_df)}")

### Q6: Test-Time Augmentation (TTA) Evaluation
Pre-pending instruction prefix 'Answer the following multiple-choice question carefully:' to prompts for the first 50 rows of test.csv and measuring Top-1 prediction stability.

In [ ]:
TTA_PREFIX = "Answer the following multiple-choice question carefully: "
q6_changed = 0

for i, row in test_df.iloc[:50].iterrows():
    orig_text = build_input(row)
    aug_text = TTA_PREFIX + build_input(row)
    orig_probs = get_probs(deb_tokenizer, deb_model, orig_text)
    aug_probs = get_probs(deb_tokenizer, deb_model, aug_text)
    tta_probs = (orig_probs + aug_probs) / 2.0
    if LABEL_MAP[np.argmax(orig_probs)] != LABEL_MAP[np.argmax(tta_probs)]:
        q6_changed += 1

print(f"Q6 Rows with different Top-1 prediction after TTA: {q6_changed}")

### DeBERTa vs. Weighted Ensemble Comparative Analysis (Q7 - Q9 WIP)
Comparing Top-1 prediction differences and measuring confidence gain on the first 100 rows of test.csv.

In [ ]:
q7_different = 0
for i, row in test_df.iloc[:100].iterrows():
    text = build_input(row)
    deb_p = get_probs(deb_tokenizer, deb_model, text)
    rob_p = get_probs(rob_tokenizer, rob_model, text)
    w_p = DEBERTA_WEIGHT * deb_p + ROBERTA_WEIGHT * rob_p
    
    deb_top1 = LABEL_MAP[np.argmax(deb_p)]
    ens_top1 = LABEL_MAP[np.argmax(w_p)]
    if deb_top1 != ens_top1:
        q7_different += 1

print(f"Q7 Top-1 prediction differences between DeBERTa and Ensemble: {q7_different}")
# TODO: confidence gain calculation needs max probability across class dimensions, fixing in next commit